# 05 ICA Cleaning

This notebook fits ICA models, stores manual ICA exclusion decisions, and writes ICA-cleaned raw derivatives.

Recommended order:

1. Select recordings.
2. Fit ICA models if they do not exist yet.
3. Iterate through recordings and mark ICA components interactively in `ica.plot_sources`.
4. Save one ICA decision per recording.
5. Write cleaned raw derivatives after all selected recordings have an ICA decision.
6. Inspect one cleaned raw file.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import matplotlib.pyplot as plt
import pandas as pd

from meeg_pipeline.cleaning import (
    fit_ica_for_recordings,
    load_filtered_raw_for_cleaning,
    load_ica,
    load_ica_decision,
    make_cleaned_raw_path,
    save_or_load_ica_decision,
    write_cleaned_raw_for_recordings,
)
from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import (
    decision_policy_for_step,
    existing_output_policy_for_step,
    find_recording,
    ica_decisions_status_to_dataframe,
    ica_overview_to_dataframe,
    iter_recordings,
    recording_label,
    recording_results_to_dataframe,
    safe_join,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Interactive plotting backend

In [ ]:
%matplotlib qt

mne.viz.set_browser_backend("qt")
mne.set_log_level("WARNING")

print("MNE browser backend:", mne.viz.get_browser_backend())


## Selection

Use single values, lists, `None`, or `"all"`.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

Default: existing outputs are not overwritten.

Examples:

    OVERWRITE_STEPS = []
    OVERWRITE_STEPS = ["ica"]
    OVERWRITE_STEPS = ["ica_decision"]
    OVERWRITE_STEPS = ["cleaned_raw"]
    OVERWRITE_STEPS = ["ica_decision", "cleaned_raw"]
    OVERWRITE_STEPS = "all"


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "ica",
            "overwrite": should_overwrite("ica", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "ica",
                OVERWRITE_STEPS,
            ),
        },
        {
            "step": "ica_decision",
            "overwrite": should_overwrite("ica_decision", OVERWRITE_STEPS),
            "policy": decision_policy_for_step(
                "ica_decision",
                OVERWRITE_STEPS,
            ),
        },
        {
            "step": "cleaned_raw",
            "overwrite": should_overwrite("cleaned_raw", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "cleaned_raw",
                OVERWRITE_STEPS,
            ),
        },
    ]
)


## Input and output overview

This table is safe to run at any point. It checks expected inputs and outputs without modifying files.


In [ ]:
ica_overview_table = ica_overview_to_dataframe(config, selected_recordings)
ica_overview_table


## ICA fitting parameters

The values come from `configs/local.yaml`.

For your current project, `decim: 4` means that MNE uses every fourth sample during ICA fitting. `fit_resample_sfreq` is ignored if `decim` is set.


In [ ]:
ICA_N_COMPONENTS = config.cleaning.ica.n_components
ICA_METHOD = config.cleaning.ica.method
ICA_RANDOM_STATE = config.cleaning.ica.random_state
ICA_MAX_ITER = config.cleaning.ica.max_iter
ICA_DECIM = config.cleaning.ica.decim
ICA_FIT_RESAMPLE_SFREQ = config.cleaning.ica.fit_resample_sfreq

pd.DataFrame(
    [
        {
            "n_components": ICA_N_COMPONENTS,
            "method": ICA_METHOD,
            "random_state": ICA_RANDOM_STATE,
            "max_iter": ICA_MAX_ITER,
            "decim": ICA_DECIM,
            "fit_resample_sfreq": ICA_FIT_RESAMPLE_SFREQ,
        }
    ]
)


## Fit ICA models

Existing ICA files are skipped by default unless `OVERWRITE_STEPS` contains `"ica"`.


In [ ]:
ica_policy = existing_output_policy_for_step(
    "ica",
    OVERWRITE_STEPS,
)

fit_results = fit_ica_for_recordings(
    config,
    selected_recordings,
    on_existing=ica_policy,
    n_components=ICA_N_COMPONENTS,
    method=ICA_METHOD,
    random_state=ICA_RANDOM_STATE,
    max_iter=ICA_MAX_ITER,
    decim=ICA_DECIM,
    fit_resample_sfreq=ICA_FIT_RESAMPLE_SFREQ,
)

recording_results_to_dataframe(selected_recordings, fit_results)


## Iterative ICA component inspection and decision saving

This step iterates through all selected recordings, similar to the bad-channel inspection workflow.

For each recording:

1. The fitted ICA and the filtered raw data are loaded.
2. `ica.plot_components()` opens as a topography overview.
3. `ica.plot_sources(raw_for_ica, block=True)` opens for interactive component inspection.
4. Click components in the source browser to mark them for exclusion. Marked components should appear grey.
5. Close the source browser window to continue.
6. The current `ica.exclude` list is saved as the ICA decision for that recording.

Existing ICA decisions are loaded and skipped by default. To revise decisions, set:

    OVERWRITE_STEPS = ["ica_decision"]

If no components should be excluded, leave all components unmarked and close the browser. An empty decision is still saved.


In [ ]:
ica_decision_policy = decision_policy_for_step(
    "ica_decision",
    OVERWRITE_STEPS,
)

interactive_decision_results = []

for index, recording in enumerate(selected_recordings):
    label = recording_label(recording)

    print("=" * 80)
    print(f"ICA inspection {index + 1}/{len(selected_recordings)}: {label}")
    print("=" * 80)

    existing_decision = load_ica_decision(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    if existing_decision.status == "loaded" and ica_decision_policy == "load":
        interactive_decision_results.append(
            {
                "index": index,
                "recording": label,
                "status": "loaded_existing",
                "message": "ICA decision already exists; GUI not opened.",
                "exclude": safe_join(existing_decision.exclude),
                "n_excluded": len(existing_decision.exclude),
                "method": existing_decision.method,
                "notes": existing_decision.notes,
                "path": existing_decision.path,
            }
        )

        print("Existing ICA decision found; GUI not opened.")
        print(f"Excluded components: {existing_decision.exclude}")
        print()
        continue

    raw_result = load_filtered_raw_for_cleaning(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        preload=True,
    )

    ica_result = load_ica(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    if raw_result.raw is None or ica_result.ica is None:
        status = raw_result.status if raw_result.raw is None else ica_result.status
        message = raw_result.message if raw_result.raw is None else ica_result.message
        path = raw_result.path if raw_result.raw is None else ica_result.path

        interactive_decision_results.append(
            {
                "index": index,
                "recording": label,
                "status": status,
                "message": message,
                "exclude": "",
                "n_excluded": None,
                "method": "",
                "notes": "",
                "path": path,
            }
        )

        print(message)
        print()
        continue

    raw_for_ica = raw_result.raw
    ica = ica_result.ica

    if existing_decision.status == "loaded" and ica_decision_policy == "overwrite":
        ica.exclude = list(existing_decision.exclude)
    else:
        ica.exclude = []

    print("Instructions:")
    print("  1. Keep the components overview window open as a topography reference.")
    print("  2. In the source browser, click component names/traces to mark exclusions.")
    print("  3. Marked components should appear grey and are stored in ica.exclude.")
    print("  4. Close the source browser window to save the current exclusion list.")
    print("  5. If no component is clearly artifactual, leave all components unmarked.")

    ica.plot_components(show=True)
    ica.plot_sources(raw_for_ica, block=True)

    selected_exclude = sorted(int(component) for component in ica.exclude)

    if selected_exclude:
        decision_notes = (
            "Rejected components marked interactively in ica.plot_sources()."
        )
    else:
        decision_notes = (
            "No clearly identifiable artifact components rejected during "
            "interactive ica.plot_sources() inspection."
        )

    decision_result = save_or_load_ica_decision(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        exclude=selected_exclude,
        notes=decision_notes,
        on_existing=ica_decision_policy,
    )

    interactive_decision_results.append(
        {
            "index": index,
            "recording": label,
            "status": decision_result.status,
            "message": decision_result.message,
            "exclude": safe_join(decision_result.exclude),
            "n_excluded": len(decision_result.exclude),
            "method": decision_result.method,
            "notes": decision_result.notes,
            "path": decision_result.path,
        }
    )

    print(f"Saved ICA decision for {label}: exclude={decision_result.exclude}")
    print()

    plt.close("all")

interactive_decision_results_table = pd.DataFrame(interactive_decision_results)
interactive_decision_results_table


## ICA decision overview

Run this after the iterative inspection. All selected recordings should have `status = loaded` before writing cleaned raw derivatives.


In [ ]:
ica_decision_overview = ica_decisions_status_to_dataframe(
    config,
    selected_recordings,
)
ica_decision_overview


## Write cleaned raw derivatives

This applies saved ICA decisions and writes `desc-cleaned_meg.fif`.

Existing cleaned raw files are skipped by default unless `OVERWRITE_STEPS` contains `"cleaned_raw"`.


In [ ]:
cleaned_policy = existing_output_policy_for_step(
    "cleaned_raw",
    OVERWRITE_STEPS,
)

cleaned_results = write_cleaned_raw_for_recordings(
    config,
    selected_recordings,
    on_existing=cleaned_policy,
)

recording_results_to_dataframe(selected_recordings, cleaned_results)


## Select one cleaned raw derivative for inspection

Use explicit BIDS entities instead of an index.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "0001"
    INSPECT_SESSION = None
    INSPECT_TASK = "example"
    INSPECT_RUN = None

    INSPECT = find_recording(
        selected_recordings,
        subject=INSPECT_SUBJECT,
        session=INSPECT_SESSION,
        task=INSPECT_TASK,
        run=INSPECT_RUN,
    )

    if INSPECT is None:
        cleaned_raw = None

        cleaned_inspect_status = pd.DataFrame(
            [
                {
                    "recording": "",
                    "status": "not_selected",
                    "message": (
                        "No matching recording found in selected_recordings. "
                        "Check INSPECT_SUBJECT/SESSION/TASK/RUN or the selection above."
                    ),
                    "path": "",
                }
            ]
        )

    else:
        cleaned_path = make_cleaned_raw_path(
            config,
            subject=INSPECT["subject"],
            session=INSPECT["session"],
            task=INSPECT["task"],
            run=INSPECT["run"],
        )

        if not cleaned_path.exists():
            cleaned_raw = None

            cleaned_inspect_status = pd.DataFrame(
                [
                    {
                        "recording": recording_label(INSPECT),
                        "status": "missing_input",
                        "message": "Cleaned raw derivative does not exist.",
                        "path": str(cleaned_path),
                    }
                ]
            )
        else:
            cleaned_raw = mne.io.read_raw_fif(
                cleaned_path,
                preload=False,
                verbose="error",
            )

            cleaned_inspect_status = pd.DataFrame(
                [
                    {
                        "recording": recording_label(INSPECT),
                        "status": "loaded",
                        "message": "",
                        "path": str(cleaned_path),
                    }
                ]
            )

    cleaned_inspect_status
else:
    print('Skipped single-file inspection cell 22 in 1B_meg_preprocessing/05_ica_cleaning.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Plot selected cleaned raw derivative

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if "cleaned_raw" not in globals() or cleaned_raw is None:
        print("No cleaned raw loaded. Run the previous inspection cell first.")
    else:
        cleaned_raw.plot(
            picks="meg",
            block=True,
        )
else:
    print('Skipped single-file inspection cell 24 in 1B_meg_preprocessing/05_ica_cleaning.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
